# Weight Break Project — Negotiation Notebook

This notebook reuses the data and outputs already prepared in the project and builds a **negotiation-ready rate card view**, starting from the matched linehaul model.

It is designed for **local Jupyter Notebook** with files stored in:

`Desktop/Carrier project`

In [ ]:
import os
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Data directory

In [ ]:
DATA_DIR = Path.home() / "Desktop" / "Carrier project"
print("DATA_DIR:", DATA_DIR)
print("Exists:", DATA_DIR.exists())
if DATA_DIR.exists():
    print("\nFiles in folder:")
    for p in sorted(DATA_DIR.iterdir()):
        print("-", p.name)

DATA_DIR: C:\Users\atb6113\Desktop\Carrier project
Exists: True

Files in folder:
- .ipynb_checkpoints
- 2025-02-19 Nunner offer  OUTBOUND version Final 1.1 Valid 2025.xlsx
- 2025-06-26 Nunner offer INBOUND version Final 3.0 valid 2025.xlsx
- 2025-07-10 ID_105852_01_EMEA_V01_BE_20250707-090556-629.xlsx
- Carrier_Optimization_Full_Project_Plan.docx
- Carrier_Optimization_Project_and_Thesis_Overview.docx
- Carrier_Selection_Detailed_Report.pdf
- Carrier_Selection_Full_Presentation.pptx
- Copy of Freight Spend CMI Belgium_Jan'25_Nov'25.xlsx
- desktop.ini
- DHL-GEODIS-NUNNER
- DHL.zip
- dhl_rate_fact_final.csv
- dhl_rate_fact_parsed.csv
- Freight Spend CMI Belgium_Jan'25_Nov'25.xlsb
- freight_config.json
- Freight_Model_Master.ipynb
- Freight_Model_Master_v2.ipynb
- Freight_Optimization_Final_Report.docx
- Freight_Spend_Optimization_Executive_Summary.docx
- Freight_Spend_Optimization_Full_Executive_Report.docx
- Geodis.zip
- geodis_rate_fact_final.csv
- Notebook_00_Run_Any_Country.ipynb
- 

## 2. Discover files

In [ ]:
xlsx_files = sorted(glob.glob(str(DATA_DIR / "*.xlsx")))
xlsb_files = sorted(glob.glob(str(DATA_DIR / "*.xlsb")))
csv_files  = sorted(glob.glob(str(DATA_DIR / "*.csv")))

print("Found .xlsx:", len(xlsx_files))
print("Found .xlsb:", len(xlsb_files))
print("Found .csv :", len(csv_files))

Found .xlsx: 7
Found .xlsb: 1
Found .csv : 8


In [ ]:
def pick_file(files, must_include=(), any_include=()):
    def score(fn):
        name = Path(fn).name.lower()
        s = 0
        for t in must_include:
            if t.lower() in name:
                s += 10
            else:
                return -1
        if any_include:
            s += sum(1 for t in any_include if t.lower() in name)
        return s

    ranked = sorted(files, key=score, reverse=True)
    best = ranked[0] if ranked and score(ranked[0]) >= 0 else None
    return best

FILES = {}
FILES["invoices"]        = pick_file(xlsb_files, must_include=("freight", "spend"), any_include=("belgium", "cmi"))
FILES["nunner_out_csv"]  = pick_file(csv_files, must_include=("nunner", "out", "rate", "fact"))
FILES["nunner_in_csv"]   = pick_file(csv_files, must_include=("nunner", "in", "rate", "fact"))
FILES["geodis_csv"]      = pick_file(csv_files, must_include=("geodis", "rate", "fact"))

for k, v in FILES.items():
    print(f"{k:15s} -> {Path(v).name if v else None}")

invoices        -> Freight Spend CMI Belgium_Jan'25_Nov'25.xlsb
nunner_out_csv  -> nunner_out_rate_fact_final_v2.csv
nunner_in_csv   -> nunner_in_rate_fact_final_v2.csv
geodis_csv      -> geodis_rate_fact_final.csv


In [ ]:
try:
    import pyxlsb  # noqa
    print("pyxlsb is installed.")
except Exception:
    print("pyxlsb is missing. Uncomment the next line once if needed.")
    # %pip install pyxlsb

pyxlsb is installed.


## 4. Load invoice workbook

In [ ]:
assert FILES["invoices"] is not None, "Invoice .xlsb file not found in DATA_DIR."

file_path = FILES["invoices"]
print("Invoice file:", file_path)

xls = pd.ExcelFile(file_path, engine="pyxlsb")
print("Sheets:", xls.sheet_names)

df_raw = pd.read_excel(file_path, sheet_name=0, engine="pyxlsb")
print("df_raw shape:", df_raw.shape)
display(df_raw.head(5))

Invoice file: C:\Users\atb6113\Desktop\Carrier project\Freight Spend CMI Belgium_Jan'25_Nov'25.xlsb
Sheets: ['FreightDataReportForAtmus']
df_raw shape: (45946, 59)


,7ueuur33333e232333r22323dwq1diiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii 8888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888 8 v5ftg5v44e5tddwxt545iiiiiiiii44444444444tr4t544a35t554553erwd6i5i ttri23sx4ie 4t4t43ugigggggggfffffffffffffffffbh cqqfivfgfvyiftvqw,Customer Reference No,Account Code,Shipment Id,Carrier Name,Carrier Code,Invoice No,Inv Net Amt,Inv Gross Amt,Origin Name,Origin ID,OriginCity,Origin Zip Code,Origin Country,Stoppage 1 City,Stoppage 1 Country,Stoppage 1 Zip,Stoppage 2 City,Stoppage 2 Country,Stoppage 2 Zip,Destination Name,Destination ID,Destination City,Destination Zip Code,Destination Country,Actual Weight,Volume,Chargeable Weight,Loading Meter,Equipment Type,Net Amount,Amount in USD,Gross Amount,Header Currency,Currency,Exchange Rate,VAT Amt,Accessorial Code,Accessorial Name,Pickup Date,Delivery Date,Unit Qty,Pallet Type,Shipment Type,Transportation Mode,Carrier Reference No,Pay Status Group Code,Pay Status Category Code,Corporate Location Name,Product Group Desc,Inbound Outbound Code Desc,Kilometers,Arrival Date,Invoice Date,Remitted Date,Service Level Type,Shipment Net Amount,Batch,Month
0,Atmus,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,502.30,521.220297,607.79,EUR,EUR,NaN,105.49,LHL,Linehaul,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2
1,Atmus,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,20.00,20.753346,24.20,EUR,EUR,NaN,4.20,CUS,Customs Fee,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2
2,Atmus,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,10.95,11.362457,13.25,EUR,EUR,NaN,2.30,MTB,MAUT BELGIUM,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2
3,Atmus,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,10.70,11.103040,12.95,EUR,EUR,NaN,2.25,MTG,Maut Germany,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2
4,Atmus,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,55.00,57.071703,66.55,EUR,EUR,NaN,11.55,DLC,Delivery Charges,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2


## 5. Clean invoice structure

In [ ]:
def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out

df = clean_cols(df_raw)

bad_first = df.columns[0]
if bad_first not in ["Customer Reference No", "Account Code", "Shipment Id"]:
    print("Dropping first non-analytical column:", bad_first)
    df = df.iloc[:, 1:].copy()

df.columns = [str(c).strip() for c in df.columns]
print(df.shape)
display(df.head(5))

Dropping first non-analytical column: 7ueuur33333e232333r22323dwq1diiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii  8888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888 8 v5ftg5v44e5tddwxt545iiiiiiiii44444444444tr4t544a35t554553erwd6i5i ttri23sx4ie 4t4t43ugigggggggfffffffffffffffffbh cqqfivfgfvyiftvqw
(45946, 58)


,Customer Reference No,Account Code,Shipment Id,Carrier Name,Carrier Code,Invoice No,Inv Net Amt,Inv Gross Amt,Origin Name,Origin ID,OriginCity,Origin Zip Code,Origin Country,Stoppage 1 City,Stoppage 1 Country,Stoppage 1 Zip,Stoppage 2 City,Stoppage 2 Country,Stoppage 2 Zip,Destination Name,Destination ID,Destination City,Destination Zip Code,Destination Country,Actual Weight,Volume,Chargeable Weight,Loading Meter,Equipment Type,Net Amount,Amount in USD,Gross Amount,Header Currency,Currency,Exchange Rate,VAT Amt,Accessorial Code,Accessorial Name,Pickup Date,Delivery Date,Unit Qty,Pallet Type,Shipment Type,Transportation Mode,Carrier Reference No,Pay Status Group Code,Pay Status Category Code,Corporate Location Name,Product Group Desc,Inbound Outbound Code Desc,Kilometers,Arrival Date,Invoice Date,Remitted Date,Service Level Type,Shipment Net Amount,Batch,Month
0,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,502.30,521.220297,607.79,EUR,EUR,NaN,105.49,LHL,Linehaul,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2
1,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,20.00,20.753346,24.20,EUR,EUR,NaN,4.20,CUS,Customs Fee,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2
2,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,10.95,11.362457,13.25,EUR,EUR,NaN,2.30,MTB,MAUT BELGIUM,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2
3,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,10.70,11.103040,12.95,EUR,EUR,NaN,2.25,MTG,Maut Germany,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2
4,4838131,'AE9ADRADR90004911023000000000000,2416814,Hamann International Logistics NV,HAMABEWETT,1466127,644.46,779.8,UFI FILTERS SPA,,Nogarole Rocca,37060,IT,NaN,NaN,NaN,NaN,NaN,NaN,CMI Filtration,CMIFBERUMS,Rumst,2840,BE,360.0,6.48,360.0,2.0,NaN,55.00,57.071703,66.55,EUR,EUR,NaN,11.55,DLC,Delivery Charges,45601,45601,5,Per Pallet,B,TR,NaN,Paid,Paid in Full,CMI Filtration Belgium,NaN,Inbound,0.0,45637,45623,45694,NaN,644.46,8,2


## 6. Build shipment-level and linehaul-level invoice facts

In [ ]:
shipment_cols = [
    "Shipment Id",
    "Carrier Name",
    "Carrier Code",
    "Invoice No",
    "Origin Country",
    "Destination Country",
    "Service Level Type",
    "Inbound Outbound Code Desc",
    "Transportation Mode",
    "Actual Weight",
    "Chargeable Weight",
    "Volume",
    "Loading Meter",
    "Pickup Date",
    "Delivery Date",
    "Invoice Date",
    "Currency"
]

shipment_fact = (
    df
    .groupby(shipment_cols, dropna=False)
    .agg(
        invoiced_net_amount=("Net Amount", "sum"),
        invoiced_gross_amount=("Gross Amount", "sum")
    )
    .reset_index()
)

linehaul_df = df[df["Accessorial Code"].astype(str).str.upper() == "LHL"].copy()
linehaul_df.columns = [str(c).strip() for c in linehaul_df.columns]

linehaul_shipments = (
    linehaul_df
    .groupby(
        shipment_cols,
        dropna=False
    )
    .agg(
        invoiced_linehaul_amount=("Net Amount", "sum")
    )
    .reset_index()
)

print("shipment_fact:", shipment_fact.shape)
print("linehaul_shipments:", linehaul_shipments.shape)
display(linehaul_shipments.head(5))

shipment_fact: (22727, 19)
linehaul_shipments: (18584, 18)


,Shipment Id,Carrier Name,Carrier Code,Invoice No,Origin Country,Destination Country,Service Level Type,Inbound Outbound Code Desc,Transportation Mode,Actual Weight,Chargeable Weight,Volume,Loading Meter,Pickup Date,Delivery Date,Invoice Date,Currency,invoiced_linehaul_amount
0,2405596,GEODIS CALBERSON LILLE EUROPE,GEODFRLOMM,2405912725,FR,FR,AFF,Outbound,LTL,963.0,963.0,0.0,0.6,45600,45600,45606,EUR,87.82
1,2405602,GEODIS CALBERSON LILLE EUROPE,GEODFRLOMM,2405912725,FR,FR,AFF,Outbound,LTL,60.0,60.0,0.0,0.4,45600,45600,45606,EUR,106.74
2,2405605,GEODIS CALBERSON LILLE EUROPE,GEODFRLOMM,2405912725,FR,FR,AFF,Outbound,LTL,295.0,295.0,0.0,0.8,45600,45600,45606,EUR,243.12
3,2405609,GEODIS CALBERSON LILLE EUROPE,GEODFRLOMM,2405912725,FR,FR,AFF,Outbound,LTL,61.0,61.0,0.0,0.4,45600,45600,45606,EUR,106.74
4,2405612,GEODIS CALBERSON LILLE EUROPE,GEODFRLOMM,2405912725,FR,FR,AFF,Outbound,LTL,1485.0,1485.0,0.0,3.3,45600,45600,45606,EUR,143.78


## 7. Focus carrier scope

In [ ]:
focus_carriers = ["NUNNER", "DHL", "GEODIS"]

linehaul_shipments_focused = linehaul_shipments[
    linehaul_shipments["Carrier Name"].astype(str).str.upper().str.contains("|".join(focus_carriers), na=False)
].copy()

print("linehaul_shipments_focused:", linehaul_shipments_focused.shape)
display(linehaul_shipments_focused["Carrier Name"].value_counts())

linehaul_shipments_focused: (16687, 18)


Carrier Name
GEODIS CALBERSON LILLE EUROPE    11168
Nunner Logistics B.V.             5519
Name: count, dtype: int64

## 8. Load prepared rate-card outputs

This notebook expects the previously generated CSVs to be available in the project folder.

In [ ]:
assert FILES["nunner_out_csv"] is not None, "Missing Nunner outbound rate CSV."
assert FILES["nunner_in_csv"] is not None, "Missing Nunner inbound rate CSV."
assert FILES["geodis_csv"] is not None, "Missing Geodis rate CSV."

nunner_out_rate_fact_final_v2 = pd.read_csv(FILES["nunner_out_csv"])
nunner_in_rate_fact_final_v2  = pd.read_csv(FILES["nunner_in_csv"])
geodis_rate_fact_final        = pd.read_csv(FILES["geodis_csv"])

print("nunner_out:", nunner_out_rate_fact_final_v2.shape)
print("nunner_in :", nunner_in_rate_fact_final_v2.shape)
print("geodis    :", geodis_rate_fact_final.shape)

nunner_out: (115010, 12)
nunner_in : (669, 14)
geodis    : (107, 11)


C:\Users\atb6113\AppData\Local\Temp\ipykernel_22996\2866790449.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  nunner_out_rate_fact_final_v2 = pd.read_csv(FILES["nunner_out_csv"])


## 9. Normalize invoice fields and create reconstructable set

DHL is part of broader descriptive benchmarking, but in this invoice extract it does not expose a clean `LHL` linehaul component comparable to Nunner and Geodis road tariffs.  
Therefore, the linehaul reconstruction and negotiation model below focuses on **Nunner and Geodis**.

In [ ]:
lh = linehaul_shipments_focused.copy()

lh["CARRIER"] = lh["Carrier Name"].astype(str).str.upper()
lh.loc[lh["CARRIER"].str.contains("NUNNER"), "CARRIER"] = "NUNNER"
lh.loc[lh["CARRIER"].str.contains("GEODIS"), "CARRIER"] = "GEODIS"
lh.loc[lh["CARRIER"].str.contains("DHL"),    "CARRIER"] = "DHL"

lh["DIRECTION"] = lh["Inbound Outbound Code Desc"].astype(str).str.upper()
lh["ORIGIN_COUNTRY"] = lh["Origin Country"].astype(str).str.upper().str.strip()
lh["DESTINATION_COUNTRY"] = lh["Destination Country"].astype(str).str.upper().str.strip()
lh["SERVICE_LEVEL"] = lh["Service Level Type"].astype(str).str.upper().str.strip()

if "Chargeable Weight" in lh.columns:
    lh["BILLABLE_WEIGHT"] = pd.to_numeric(lh["Chargeable Weight"], errors="coerce")
else:
    lh["BILLABLE_WEIGHT"] = pd.to_numeric(lh["Actual Weight"], errors="coerce")

lh_recon = lh[lh["CARRIER"].isin(["NUNNER", "GEODIS"])].copy()

print("lh_recon:", lh_recon.shape)
display(lh_recon["CARRIER"].value_counts())
display(lh_recon[["CARRIER","DIRECTION","ORIGIN_COUNTRY","DESTINATION_COUNTRY","SERVICE_LEVEL","BILLABLE_WEIGHT"]].head(5))

lh_recon: (16687, 24)


CARRIER
GEODIS    11168
NUNNER     5519
Name: count, dtype: int64

,CARRIER,DIRECTION,ORIGIN_COUNTRY,DESTINATION_COUNTRY,SERVICE_LEVEL,BILLABLE_WEIGHT
0,GEODIS,OUTBOUND,FR,FR,AFF,963.0
1,GEODIS,OUTBOUND,FR,FR,AFF,60.0
2,GEODIS,OUTBOUND,FR,FR,AFF,295.0
3,GEODIS,OUTBOUND,FR,FR,AFF,61.0
4,GEODIS,OUTBOUND,FR,FR,AFF,1485.0


## 10. Build unified rate universe

In [ ]:
def prep_rates(df, carrier, direction):
    out = df.copy()
    out["CARRIER"] = carrier
    out["DIRECTION"] = direction
    for c in ["ORIGIN_COUNTRY","DESTINATION_COUNTRY"]:
        if c in out.columns:
            out[c] = out[c].astype(str).str.upper().str.strip()
        else:
            out[c] = pd.NA
    if "SERVICE_LEVEL" not in out.columns:
        out["SERVICE_LEVEL"] = pd.NA
    return out

rates_out = prep_rates(nunner_out_rate_fact_final_v2, "NUNNER", "OUTBOUND")
rates_in  = prep_rates(nunner_in_rate_fact_final_v2,  "NUNNER", "INBOUND")
rates_geo = prep_rates(geodis_rate_fact_final,        "GEODIS", "OUTBOUND")

rate_universe = pd.concat([rates_out, rates_in, rates_geo], ignore_index=True)
print("rate_universe:", rate_universe.shape)
display(rate_universe[["CARRIER","DIRECTION"]].value_counts())

rate_universe: (115786, 16)


CARRIER  DIRECTION
NUNNER   OUTBOUND     115010
         INBOUND         669
GEODIS   OUTBOUND        107
Name: count, dtype: int64

## 11. Match shipments to rate cards and build baseline

In [ ]:
def match_weight_bands(ship_df, rate_df, keys, weight_col="BILLABLE_WEIGHT"):
    m = ship_df.merge(rate_df, on=keys, how="left", suffixes=("", "_RATE"))
    m = m[(m[weight_col] >= m["WEIGHT_FROM"]) & (m[weight_col] < m["WEIGHT_TO"])].copy()
    return m

lh_nunner_out = lh_recon[(lh_recon["CARRIER"]=="NUNNER") & (lh_recon["DIRECTION"]=="OUTBOUND")].copy()
rc_nunner_out = rate_universe[(rate_universe["CARRIER"]=="NUNNER") & (rate_universe["DIRECTION"]=="OUTBOUND")].copy()
matched_nunner_out = match_weight_bands(lh_nunner_out, rc_nunner_out, keys=["CARRIER","DIRECTION","DESTINATION_COUNTRY"])

lh_nunner_in = lh_recon[(lh_recon["CARRIER"]=="NUNNER") & (lh_recon["DIRECTION"]=="INBOUND")].copy()
rc_nunner_in = rate_universe[(rate_universe["CARRIER"]=="NUNNER") & (rate_universe["DIRECTION"]=="INBOUND")].copy()
matched_nunner_in = match_weight_bands(lh_nunner_in, rc_nunner_in, keys=["CARRIER","DIRECTION","ORIGIN_COUNTRY","DESTINATION_COUNTRY"])

lh_geodis = lh_recon[(lh_recon["CARRIER"]=="GEODIS") & (lh_recon["DIRECTION"]=="OUTBOUND")].copy()
rc_geodis = rate_universe[(rate_universe["CARRIER"]=="GEODIS") & (rate_universe["DIRECTION"]=="OUTBOUND")].copy()
matched_geodis = match_weight_bands(lh_geodis, rc_geodis, keys=["CARRIER","DIRECTION"])

matched_all = pd.concat([matched_nunner_out, matched_nunner_in, matched_geodis], ignore_index=True)
print("matched_all:", matched_all.shape)

matched_all["SHIP_KEY"] = (
    matched_all["Shipment Id"].astype(str) + "|" +
    matched_all["Invoice No"].astype(str) + "|" +
    matched_all["CARRIER"].astype(str)
)

best_match = (
    matched_all.sort_values(["SHIP_KEY","RATE"], ascending=[True, True])
               .drop_duplicates("SHIP_KEY", keep="first")
               .copy()
)

best_match["expected_linehaul_cost"] = pd.to_numeric(best_match["RATE"], errors="coerce")
best_match["delta"] = best_match["invoiced_linehaul_amount"] - best_match["expected_linehaul_cost"]
best_match["abs_delta"] = best_match["delta"].abs()

print("best_match:", best_match.shape)
display(best_match[["CARRIER","DIRECTION","BILLABLE_WEIGHT","invoiced_linehaul_amount","expected_linehaul_cost","delta"]].head(10))

matched_all: (208084, 38)
best_match: (15285, 42)


,CARRIER,DIRECTION,BILLABLE_WEIGHT,invoiced_linehaul_amount,expected_linehaul_cost,delta
168339,GEODIS,OUTBOUND,963.0,87.82,340.68,-252.86
168341,GEODIS,OUTBOUND,60.0,106.74,42.12,64.62
168345,GEODIS,OUTBOUND,295.0,243.12,153.00,90.12
168347,GEODIS,OUTBOUND,61.0,106.74,42.12,64.62
168351,GEODIS,OUTBOUND,392.0,143.78,193.80,-50.02
168353,GEODIS,OUTBOUND,55.0,131.20,42.12,89.08
168357,GEODIS,OUTBOUND,530.0,143.78,231.54,-87.76
168359,GEODIS,OUTBOUND,1044.0,143.78,349.86,-206.08
168361,GEODIS,OUTBOUND,780.0,414.48,274.38,140.10
168363,GEODIS,OUTBOUND,849.0,498.02,314.16,183.86


## 12. Coverage and gap diagnostics

In [ ]:
lh_attempt = lh_recon.copy()
lh_attempt["SHIP_KEY"] = (
    lh_attempt["Shipment Id"].astype(str) + "|" +
    lh_attempt["Invoice No"].astype(str) + "|" +
    lh_attempt["CARRIER"].astype(str)
)

covered = lh_attempt.merge(
    best_match[["SHIP_KEY","expected_linehaul_cost"]],
    on="SHIP_KEY",
    how="left"
)

coverage = (
    covered.assign(matched=lambda x: x["expected_linehaul_cost"].notna())
           .groupby(["CARRIER","DIRECTION"])["matched"]
           .agg(total="count", matched="sum")
           .reset_index()
)
coverage["match_rate"] = coverage["matched"] / coverage["total"]

gap_by_carrier = coverage.copy()
gap_by_carrier["unmatched"] = gap_by_carrier["total"] - gap_by_carrier["matched"]
gap_by_carrier["unmatched_rate"] = gap_by_carrier["unmatched"] / gap_by_carrier["total"]

display(coverage)
display(gap_by_carrier)

,CARRIER,DIRECTION,total,matched,match_rate
0,GEODIS,INBOUND,13,0,0.000000
1,GEODIS,OUTBOUND,11155,11073,0.992649
2,NUNNER,INBOUND,513,429,0.836257
3,NUNNER,OUTBOUND,5006,3783,0.755693


,CARRIER,DIRECTION,total,matched,match_rate,unmatched,unmatched_rate
0,GEODIS,INBOUND,13,0,0.000000,13,1.000000
1,GEODIS,OUTBOUND,11155,11073,0.992649,82,0.007351
2,NUNNER,INBOUND,513,429,0.836257,84,0.163743
3,NUNNER,OUTBOUND,5006,3783,0.755693,1223,0.244307


## 13. Negotiation-ready target card — Geodis outbound

This section converts the matched Geodis outbound baseline into a negotiation table.
The logic is:
- identify breaks with high shipment concentration
- highlight breaks with positive total delta
- propose a target rate for those breaks

In [ ]:
geo = best_match[
    (best_match["CARRIER"]=="GEODIS") &
    (best_match["DIRECTION"]=="OUTBOUND")
].copy()

geo_target = (
    geo.groupby(
        ["CARRIER","DIRECTION","ORIGIN_COUNTRY","DESTINATION_COUNTRY","WEIGHT_FROM","WEIGHT_TO"],
        dropna=False
    )
    .agg(
        shipment_count=("SHIP_KEY","count"),
        avg_billable_weight=("BILLABLE_WEIGHT","mean"),
        current_rate=("expected_linehaul_cost","mean"),
        avg_invoiced=("invoiced_linehaul_amount","mean"),
        total_delta=("delta","sum"),
        pct_over_expected=("delta", lambda s: (s > 0).mean())
    )
    .reset_index()
)

# Negotiation proposal rule:
# If a break shows positive total delta, propose a 5% reduction to current rate.
geo_target["proposed_rate"] = np.where(
    geo_target["total_delta"] > 0,
    geo_target["current_rate"] * 0.95,
    geo_target["current_rate"]
)

geo_target["priority_score"] = geo_target["shipment_count"] * geo_target["total_delta"].clip(lower=0)

geo_target = geo_target.sort_values("priority_score", ascending=False).reset_index(drop=True)

display(geo_target.head(25))

,CARRIER,DIRECTION,ORIGIN_COUNTRY,DESTINATION_COUNTRY,WEIGHT_FROM,WEIGHT_TO,shipment_count,avg_billable_weight,current_rate,avg_invoiced,total_delta,pct_over_expected,proposed_rate,priority_score
0,GEODIS,OUTBOUND,BE,FR,0.00,16.75,6042,3.784086,12.26,18.899975,40118.73,0.962595,11.6470,2.423974e+08
1,GEODIS,OUTBOUND,FR,FR,0.00,289.68,556,187.211942,67.32,152.717086,47480.78,0.676259,63.9540,2.639931e+07
2,GEODIS,OUTBOUND,FR,FR,69.96,93.06,331,80.495831,51.41,146.074562,31333.97,0.740181,48.8395,1.037154e+07
3,GEODIS,OUTBOUND,BE,FR,0.00,289.68,445,178.757416,67.32,88.496562,9423.57,0.647191,63.9540,4.193489e+06
4,GEODIS,OUTBOUND,FR,FR,40.24,69.96,214,61.144439,42.12,106.704393,13821.06,0.542056,40.0140,2.957707e+06
5,GEODIS,OUTBOUND,BE,FR,16.75,25.22,441,20.782177,19.01,26.966168,3508.67,0.913832,18.0595,1.547323e+06
6,GEODIS,OUTBOUND,FR,FR,93.06,112.44,144,101.642431,59.11,127.418889,9836.48,0.604167,56.1545,1.416453e+06
7,GEODIS,OUTBOUND,FR,FR,0.00,16.75,87,7.464253,12.26,168.167126,13563.92,0.954023,11.6470,1.180061e+06
8,GEODIS,OUTBOUND,BE,FR,0.00,40.24,246,36.712276,25.59,36.853008,2770.70,0.813008,24.3105,6.815922e+05
9,GEODIS,OUTBOUND,FR,FR,16.75,25.22,47,21.394255,19.01,161.232766,6684.47,0.936170,18.0595,3.141701e+05


## 14. Export negotiation target card

In [ ]:
out_path = DATA_DIR / "geodis_negotiation_target_card.csv"
geo_target.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: C:\Users\atb6113\Desktop\Carrier project\geodis_negotiation_target_card.csv
